## Numerical Analysis - Spring semester 2026
# Serie 09 - Gradient methods for symmetric positive-definite linear systems

First, we will need to import some of the usual packages. You will have to run this cell every time you restart your notebook.

In [ ]:
import numpy
import copy
import time
import matplotlib.pyplot

<hr style="clear:both">

### Exercise 1: Gaussian elimination without pivoting

<div class="alert alert-success">
    
**Exercise 1:** Using only `numpy`, implement the Python function `GE(A, b)`, which takes as input arguments a two-dimensional array `A` and a one-dimensional
array `b`, and returns the one-dimensional array obtained by applying Algorithm 2.2 of the lecture notes (without the `if` statement) and then back substitution. The output should overwrite `b`.
</div>


In [ ]:
# Gaussian elimination without pivoting
def GE(A, b):
   n = len(A)
   for i in range(n-1):
      A[1+i:n, i] = A[1+i:n, i]/A[i, i]
      A[1+i:n, 1+i:n] = A[1+i:n, 1+i:n]-numpy.outer(A[1+i:n, i], A[i, 1+i:n])
      b[1+i:n] = b[1+i:n]-A[1+i:n, i]*b[i]
   for i in range(n-1, -1, -1):
      b[i] = (b[i]-A[i, i+1:n].dot(b[i+1:n]))/A[i, i]
   return b

<hr style="clear:both">

### Exercise 2: Steepest descent and conjugate gradient method

<div class="alert alert-success">
    
**Exercise 2:** Using only numpy, implement the Python functions `SD(A, b, x)` and `CG(A, b, x)`, which take as input arguments a two-dimensional array `A` and two one-dimensional arrays `b` and `x`, and return the one-dimensional array obtained by applying, respectively, Algorithm 4.1 of the lecture notes, stopping when the squared 2-norm of `b-A.dot(x)` is smaller than or equal to 10<sup>−15</sup>, and Algorithm 4.3 of the lecture notes, stopping when `len(b)` iterations have been performed or the squared 2-norm of `b-A.dot(x)` is smaller than or equal to 10<sup>−15</sup>.
</div>

In [ ]:
# Steepest descent with exact line search
def SD(A, b, x):
   b = b-A.dot(x)
   s = b.dot(b)
   while s > 1e-15:
      z = A.dot(b)
      alpha = s/(b.dot(z))
      x = x+alpha*b
      b = b-alpha*z
      s = b.dot(b)
   return x

# Conjugate gradient method with exact line search
def CG(A, b, x):
   b = b-A.dot(x)
   s = b.dot(b)
   p = b
   n = len(b)
   i = 0
   while s > 1e-15 and i < n:
      z = A.dot(p)
      alpha = s/(p.dot(z))
      x = x+alpha*p
      b = b-alpha*z
      s_new = b.dot(b)
      beta = s_new/s
      s = s_new
      p = b+beta*p
      i = i+1
   return x

<hr style="clear:both">

### Exercise 3: Steepest descent is slow

<div class="alert alert-success">
    
**Exercise 3:** Explain and run the following code in Python and comment the result.
</div>

In [ ]:
# SD is slow
n = 10
n_runs = 20
TimeGE = numpy.zeros(n_runs)
TimeSD = numpy.zeros(n_runs)
TimeCG = numpy.zeros(n_runs)
ErrorGE = numpy.zeros(n_runs)
ErrorSD = numpy.zeros(n_runs)
ErrorCG = numpy.zeros(n_runs)
for i in range(n_runs):
    A = numpy.random.randn(n, n)
    A = A@numpy.transpose(A)
    x = numpy.random.randn(n)
    b = A.dot(x)
    b_copy = copy.deepcopy(b)
    TimeSD[i] = time.time()
    xSD = SD(A, b_copy, b_copy)
    TimeSD[i] = time.time()-TimeSD[i]
    ErrorSD[i] = numpy.linalg.norm(x-xSD)/numpy.linalg.norm(x)
    b_copy = copy.deepcopy(b)
    TimeCG[i] = time.time()
    xCG = CG(A, b_copy, b_copy)
    TimeCG[i] = time.time()-TimeCG[i]
    ErrorCG[i] = numpy.linalg.norm(x-xCG)/numpy.linalg.norm(x)
    b_copy = copy.deepcopy(b)
    TimeGE[i] = time.time()
    xGE = GE(A, b_copy)
    TimeGE[i] = time.time()-TimeGE[i]
    ErrorGE[i] = numpy.linalg.norm(x-xGE)/numpy.linalg.norm(x)
print('Run time of GE: ', [min(TimeGE), numpy.median(TimeGE), max(TimeGE)])
print('Run time of SD: ', [min(TimeSD), numpy.median(TimeSD), max(TimeSD)])
print('Run time of CG: ', [min(TimeCG), numpy.median(TimeCG), max(TimeCG)])
print('Error committed by GE: ', [min(ErrorGE), numpy.median(ErrorGE), max(ErrorGE)])
print('Error committed by SD: ', [min(ErrorSD), numpy.median(ErrorSD), max(ErrorSD)])
print('Error committed by CG: ', [min(ErrorCG), numpy.median(ErrorCG), max(ErrorCG)])

<hr style="clear:both">

### Exercise 4: Gaussian elimination vs conjugate gradient method on dense matrices of increasing order

<div class="alert alert-success">
    
**Exercise 4:** Explain and run the following code in Python and comment the result.
</div>

In [ ]:
# GE vs CG on dense matrices of increasing order
n_len = 20
n = 100*numpy.arange(1, n_len+1)
TimeGE = numpy.zeros(n_len)
TimeCG = numpy.zeros(n_len)
ErrorGE = numpy.zeros(n_len)
ErrorCG = numpy.zeros(n_len)
for i in range(n_len):
    A = numpy.random.randn(n[i], n[i])
    A = A@numpy.transpose(A)
    x = numpy.random.randn(n[i])
    b = A.dot(x)
    b_copy = copy.deepcopy(b)
    TimeCG[i] = time.time()
    xCG = CG(A, b, b)
    TimeCG[i] = time.time()-TimeCG[i]
    ErrorCG[i] = numpy.linalg.norm(x-xCG)/numpy.linalg.norm(x)
    TimeGE[i] = time.time()
    xGE = GE(A, b_copy)
    TimeGE[i] = time.time()-TimeGE[i]
    ErrorGE[i] = numpy.linalg.norm(x-xGE)/numpy.linalg.norm(x)
matplotlib.pyplot.figure()
matplotlib.pyplot.plot(n, numpy.log10(TimeGE), 'ro', label='GE')
matplotlib.pyplot.plot(n, numpy.log10(TimeCG), 'bo', label='CG')
matplotlib.pyplot.xlabel('$n$')
matplotlib.pyplot.ylabel('$\log_{10}(t_n)$')
matplotlib.pyplot.legend()
matplotlib.pyplot.figure()
matplotlib.pyplot.plot(n, numpy.log10(ErrorGE), 'ro', label='GE')
matplotlib.pyplot.plot(n, numpy.log10(ErrorCG), 'bo', label='CG')
matplotlib.pyplot.xlabel('$n$')
matplotlib.pyplot.ylabel('$\log_{10}(e_n)$')
matplotlib.pyplot.legend()

<hr style="clear:both">

### Exercise 5: Gaussian elimination vs conjugate gradient method on tridiagonal matrices of increasing order

<div class="alert alert-success">
    
**Exercise 5:** Explain and run the following code in Python and comment the result.
</div>

In [ ]:
# GE vs CG on tridiagonal matrices of increasing order
n_len = 20
n = 100*numpy.arange(1, n_len+1)
TimeGE = numpy.zeros(n_len)
TimeCG = numpy.zeros(n_len)
ErrorGE = numpy.zeros(n_len)
ErrorCG = numpy.zeros(n_len)
for i in range(n_len):
    u = numpy.random.randn(n[i])
    v = numpy.random.randn(n[i]-1)
    A = numpy.diag(u)+numpy.diag(v, 1)
    A = A@numpy.transpose(A)
    x = numpy.random.randn(n[i])
    b = A.dot(x)
    b_copy = copy.deepcopy(b)
    TimeCG[i] = time.time()
    xCG = CG(A, b, b)
    TimeCG[i] = time.time()-TimeCG[i]
    ErrorCG[i] = numpy.linalg.norm(x-xCG)/numpy.linalg.norm(x)
    TimeGE[i] = time.time()
    xGE = GE(A, b_copy)
    TimeGE[i] = time.time()-TimeGE[i]
    ErrorGE[i] = numpy.linalg.norm(x-xGE)/numpy.linalg.norm(x)
matplotlib.pyplot.figure()
matplotlib.pyplot.plot(n, numpy.log10(TimeGE), 'ro', label='GE')
matplotlib.pyplot.plot(n, numpy.log10(TimeCG), 'bo', label='CG')
matplotlib.pyplot.xlabel('$n$')
matplotlib.pyplot.ylabel('$\log_{10}(t_n)$')
matplotlib.pyplot.legend()
matplotlib.pyplot.figure()
matplotlib.pyplot.plot(n, numpy.log10(ErrorGE), 'ro', label='GE')
matplotlib.pyplot.plot(n, numpy.log10(ErrorCG), 'bo', label='CG')
matplotlib.pyplot.xlabel('$n$')
matplotlib.pyplot.ylabel('$\log_{10}(e_n)$')
matplotlib.pyplot.legend()

<hr style="clear:both">

## The end

Congratulations! You have made it to the end of the third exercise notebook. 